# Exercise 1

## Question 1 ##
Use scipy.stats.norm to generate 20 arrays, each containing 200 independent, identically distributed, normal random variables. Calculate and print the mean and the standard deviation of the samples in each of these arrays.

In [1]:
import numpy as np
from scipy.stats import norm

N = 200
K = 20

container_M = []
container_STD = []
container_X = []
# Your code goes here
for arr in range(K):
    X = norm.rvs(size=N)
    container_X.append(X)
    container_M.append(X.mean())
    container_STD.append(X.std(ddof=1))

print("Sample mean:")
for m in container_M:
    print("% 6.2f" % m, end="")
print("\nSample standard deviation:")
for m in container_STD:
    print("% 6.2f" % m, end="")

Sample mean:
  0.01  0.22  0.01 -0.04 -0.01 -0.10 -0.00  0.02 -0.01 -0.03  0.03 -0.05 -0.03  0.06  0.00  0.12 -0.10 -0.02  0.07  0.14
Sample standard deviation:
  1.01  0.91  0.96  1.05  0.92  0.94  0.98  1.07  0.96  0.91  1.02  1.01  0.96  0.98  1.00  0.92  0.98  0.97  0.99  0.92

## Question 2 ##
Perform the Student's t-test on each of these samples with the null hypothesis
  
$H_0$: The sample comes from a distribution of 0 mean.


Calculate the p-value of these tests. If we reject $H_0$ when the p-value is less than 5%, in how many cases will we reject $H_0$?

# Student's t-Distribution — Summary

## Why it exists

The normal distribution works well when you **know** the true population standard deviation (σ). In practice, you almost never know σ — you only have a sample, and must **estimate** it (using the sample standard deviation, *s*). This estimation introduces extra uncertainty, especially with small samples.

The **t-distribution** is what the normal distribution becomes once you replace the true (unknown) σ with an estimate from limited data.

## Shape

- Symmetric and bell-shaped, like the normal distribution.
- **Heavier tails** — more probability assigned to extreme values, reflecting the extra uncertainty of estimating σ from data.
- As sample size grows, the t-distribution converges to the standard normal distribution.

## Degrees of freedom (df)

The single parameter that defines a given t-distribution:

$$df = n - 1$$

where *n* is the sample size. The "−1" reflects that one piece of information (the sample mean) was already used to compute the sample standard deviation.

| df | Behavior |
|---|---|
| Small (e.g. < 30) | Heavier tails, more spread out, more conservative (wider) conclusions |
| Large (df → ∞) | Nearly identical to the standard normal distribution |

## Where it's used

1. **Confidence intervals for a mean**, when σ is unknown:
$$\bar{x} \pm t_{\alpha/2,\, df} \cdot \frac{s}{\sqrt{n}}$$
2. **The t-test** — comparing means (one-sample, two-sample, or paired).
3. **Regression coefficients** — significance of each coefficient is typically assessed via a t-statistic.

## The one-sample t-test

**Question it answers:** "If the true mean really were some hypothesized value μ₀, how surprising is the sample mean I actually observed?"

**Test statistic:**
$$t = \frac{\bar{X} - \mu_0}{s / \sqrt{N}}$$

- $\bar{X}$ = sample mean
- $\mu_0$ = hypothesized mean under $H_0$
- $s$ = sample standard deviation ($ddof=1$)
- $N$ = sample size

This measures: *how many standard errors away from μ₀ is the sample mean?*

**p-value (two-sided test):**
$$p = 2 \times P(T_{df} > |t|)$$

The combined probability, under $H_0$, of observing a t-statistic at least as extreme as the one calculated — in either direction.

## Significance level (α) and decision rule

- **α = 0.05** is a conventional threshold — the rate of false positives you're willing to tolerate, not a guarantee of truth.
- **Reject $H_0$** if p-value < α.
- **Fail to reject $H_0$** if p-value ≥ α (this does **not** prove $H_0$ is true — only that there isn't enough evidence against it).

### Type I error

If $H_0$ is actually true, you will still incorrectly reject it about **α × 100%** of the time, purely due to random sampling noise. E.g., across 20 independent tests where $H_0$ is true, you'd expect ~1 false rejection on average (20 × 0.05 = 1), with the actual count varying from one batch of trials to the next (it follows a Binomial(20, 0.05) distribution).

## Python (`scipy.stats`)

```python
from scipy.stats import t, ttest_1samp

# Direct distribution functions (same interface as scipy.stats.norm, plus df)
t.pdf(x, df)        # probability density at x
t.cdf(x, df)         # P(T <= x)
t.sf(x, df)          # P(T > x)  -- more numerically stable for tail probabilities
t.ppf(p, df)         # critical value: x such that P(T <= x) = p
t.rvs(df, size=N)    # random samples

# One-sample t-test (full hypothesis test in one call)
result = ttest_1samp(sample_data, popmean=0)
result.statistic     # the t-statistic
result.pvalue         # the p-value
```

### Worked example: one-sample t-test

```python
import numpy as np
from scipy.stats import ttest_1samp

X = np.random.normal(loc=0, scale=1, size=200)

result = ttest_1samp(X, popmean=0)

alpha = 0.05
if result.pvalue < alpha:
    print(f"Reject H0 (p={result.pvalue:.4f})")
else:
    print(f"Fail to reject H0 (p={result.pvalue:.4f})")
```

### Worked example: 95% confidence interval for the mean

```python
import numpy as np
from scipy.stats import t

N = len(X)
df = N - 1
sample_mean = X.mean()
sample_std = X.std(ddof=1)

alpha = 0.05
t_crit = t.ppf(1 - alpha/2, df)
margin = t_crit * (sample_std / np.sqrt(N))

ci_lower, ci_upper = sample_mean - margin, sample_mean + margin
print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
```

In [6]:
from scipy.stats import t, ttest_1samp

counterRejectH0 = 0

for X in container_X:
    result = ttest_1samp(X, popmean=0)
    if result.pvalue < 0.05:
        counterRejectH0 += 1
print("H0 was rejected ", counterRejectH0, " times.")

H0 was rejected  2  times.
